# 08. 베이스라인: ASR + 키워드 규칙 + 상태머신 — **Colab GPU**

```
음성 → ASR (단어별 시각) → 키워드 규칙으로 '일' 추출 → 상태머신 → 질의 시각별 B-S-O → 평가
```

- ASR만 바꿔 두 가지 베이스라인을 낸다: **Whisper large-v3** / **Qwen3-ASR-1.7B (+ ForcedAligner)**
  - Qwen3-ASR은 데이터 합성에 쓴 Qwen3-TTS와 같은 계열이라 합성 음성에 유리할 수 있음 → 결과 해석 때 감안
- 일부러 **단순한** 기준선. 규칙은 템플릿 목록에 맞추지 않고 일반적인 중계 표현 수준으로 만든다
- 판정을 잘못 세도 카운트 언급("투앤원")·타자 소개("원아웃, 타석에는")가 나오면 상태머신이 바로잡는다

**준비**: Drive `kbo-listen/dataset_pilot_{VERSION}/` (06 결과), Colab `/content/kbolisten/` 업로드

`qwen-asr`과 TTS용 `qwen-tts`는 요구하는 라이브러리 버전이 다를 수 있어서, TTS 노트북과는 **다른 런타임 세션**에서 돌린다

In [36]:
ASR = "qwen3"  # "whisper" 또는 "qwen3"
VERSION = "v3"   # 평가할 파일럿 버전

!pip install -q jiwer
if ASR == "qwen3":
    !pip install -q -U qwen-asr

In [37]:
import json
import re
import time
from pathlib import Path

import jiwer
import soundfile as sf
import torch
from google.colab import drive

from kbolisten.evaluate import evaluate, load_groups
from kbolisten.state_machine import answer_queries

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kbo-listen")
DATA = DRIVE / ("dataset_pilot" if VERSION == "v1" else f"dataset_pilot_{VERSION}")
OUT = DRIVE / f"baseline_{ASR}_{VERSION}"
OUT.mkdir(exist_ok=True)

queries = json.loads((DATA / "queries.json").read_text(encoding="utf-8"))
gt = json.loads((DATA / "gt.json").read_text(encoding="utf-8"))
groups = load_groups(DATA / "meta.jsonl")
print(ASR, "/", DATA.name, len(queries), "클립", sum(len(ts) for ts in queries.values()), "질의")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
qwen3 / dataset_pilot_v3 20 클립 309 질의


In [38]:
# 1) 받아쓰기. 어느 ASR이든 transcribe(path) → (전체 텍스트, [{text, start, end}, ...]) 로 맞춘다
#    이미 받아쓴 결과(transcripts.json)가 있으면 모델을 올리지 않고 그대로 쓴다 (규칙만 고쳐서 다시 평가할 때)
TRANSCRIPTS = OUT / "transcripts.json"
REUSE = True

if REUSE and TRANSCRIPTS.exists():
    transcripts = json.loads(TRANSCRIPTS.read_text(encoding="utf-8"))
    print("저장된 받아쓰기 사용:", TRANSCRIPTS)
else:
    if ASR == "whisper":
        from transformers import pipeline

        asr = pipeline(
            "automatic-speech-recognition", model="openai/whisper-large-v3",
            dtype=torch.float16, device="cuda:0",
        )

        def transcribe(path):
            out = asr(
                str(path), return_timestamps="word", chunk_length_s=30, batch_size=2,
                generate_kwargs={"language": "korean", "task": "transcribe"},
            )
            words = []
            for ch in out["chunks"]:
                start, end = ch["timestamp"]
                words.append({"text": ch["text"], "start": start, "end": end if end is not None else start})
            return out["text"], words

    else:
        from qwen_asr import Qwen3ASRModel

        asr = Qwen3ASRModel.from_pretrained(
            "Qwen/Qwen3-ASR-1.7B", dtype=torch.bfloat16, device_map="cuda:0",
            max_inference_batch_size=8, max_new_tokens=1024,  # 1~2분 클립이라 넉넉하게
            forced_aligner="Qwen/Qwen3-ForcedAligner-0.6B",   # 단어별 시각은 정렬 모델이 담당
            forced_aligner_kwargs=dict(dtype=torch.bfloat16, device_map="cuda:0"),
        )

        def flat(items):
            for x in items or []:
                if isinstance(x, (list, tuple)):
                    yield from flat(x)
                else:
                    yield x

        def transcribe(path):
            r = asr.transcribe(audio=str(path), language="Korean", return_time_stamps=True)[0]
            stamps = list(flat(r.time_stamps))
            # 시각 단위가 밀리초로 오는 경우 대비: 마지막 시각이 음성 길이보다 훨씬 크면 초로 바꿈
            scale = 1000 if stamps and stamps[-1].end_time > 1.5 * sf.info(path).duration else 1
            # 정렬 단위는 원문보다 잘게 쪼개져 있다 ("공격" "입니다"). 띄어쓰기는 규칙 셀에서 원문 기준으로 복원
            words = [{"text": s.text, "start": s.start_time / scale, "end": s.end_time / scale} for s in stamps]
            return r.text, words

    transcripts = {}
    t0 = time.perf_counter()
    for cid in queries:
        text, words = transcribe(DATA / "audio" / f"{cid}.wav")
        transcripts[cid] = {"text": text, "words": words}
    print(f"받아쓰기 {time.perf_counter() - t0:.0f}초")
    TRANSCRIPTS.write_text(json.dumps(transcripts, ensure_ascii=False, indent=1), encoding="utf-8")

first = transcripts[next(iter(transcripts))]
print(first["text"][:300])
print(first["words"][:5])

저장된 받아쓰기 사용: /content/drive/MyDrive/kbo-listen/baseline_qwen3_v3/transcripts.json
8회말 기아 공격입니다. 스코어 롯데 5대 기아 5. 마운드에는 김상수, 노아웃 타석에는 이우성, 슬라이더 먹혔고 파울입니다. 투심 왼쪽 파울입니다. 노볼 투 스트라이크입니다. 3구 낮은 공 골라냅니다. 볼 카운트 원볼 투 스트라이크. 왼쪽 파울입니다. 볼 카운트 원앤투 145 킬로 직구 몸쪽 볼입니다. 골라냅니다. 풀카운트입니다. 선구안이 좋네요. 뜬 공 잡았습니다. 원아웃 타석에는 김태군. 자 초구 일단 빠집니다. 볼 카운트 원볼. 제고가 조금 흔들리는 모습이에요. 볼이고요. 자 볼이고요. 쓰리볼입니다. 낮은 스트라이크입니다. 
[{'text': ' 8회말', 'start': 0.56, 'end': 1.04}, {'text': ' 기아', 'start': 1.44, 'end': 1.76}, {'text': ' 공격', 'start': 1.76, 'end': 2.08}, {'text': ' 입니다', 'start': 2.08, 'end': 2.4}, {'text': ' 스코어', 'start': 2.72, 'end': 3.36}]


In [39]:
# 3) 받아쓰기 품질: 실제 대본과 비교한 CER (글자 오류율). 띄어쓰기·문장부호는 빼고 비교
def norm(text):
    return re.sub(r"[\s.,!?]", "", text)


scripts = {}
with (DATA / "scripts.jsonl").open(encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        scripts[s["clip_id"]] = " ".join(u["text"] for u in s["utts"])

cer = jiwer.cer([norm(scripts[c]) for c in queries], [norm(transcripts[c]["text"]) for c in queries])
print(f"CER {cer:.2%}")

CER 4.58%


In [40]:
# 4) 키워드 규칙: 위에서부터 우선. 앞 규칙에 걸린 글자는 뒤 규칙이 다시 쓰지 않는다
#    (예: '볼카운트 투앤원'의 '볼'을 볼 판정으로 세지 않도록 카운트 규칙이 먼저 가져감)
NUM = {"노": 0, "원": 1, "투": 2, "쓰리": 3, "쓰": 3, "0": 0, "1": 1, "2": 2, "3": 3}
N = r"(노|원|투|쓰리|쓰|[0-3])"

RULES = [
    # 중계에서 직접 들은 상태 → 보정
    (r"풀\s?카운트", lambda m: ("set_count", {"b": 3, "s": 2})),
    (N + r"\s?(?:앤|-)\s?" + N, lambda m: ("set_count", {"b": NUM[m[1]], "s": NUM[m[2]]})),
    (N + r"\s?볼\s?" + N + r"\s?스트라이크", lambda m: ("set_count", {"b": NUM[m[1]], "s": NUM[m[2]]})),
    (r"(원|투|쓰리|[1-3])\s?볼(?!넷)", lambda m: ("set_count", {"b": NUM[m[1]], "s": 0})),  # '투볼' = 2-0 관례
    (N + r"\s?아웃", lambda m: ("set_outs", {"o": NUM[m[1]]})),
    (r"타석에", lambda m: ("new_batter", {})),
    # 타석 종료
    (r"(?:헛\s?스윙|루킹|스트라이크)?[,\s]*(?:삼진|3진)", lambda m: ("in_play", {"outs": 1})),
    (r"볼넷|고의\s?(?:4구|사구)|몸에\s?맞|사구|걸어\s?나", lambda m: ("walk", {})),
    (r"삼중살|트리플\s?플레이", lambda m: ("in_play", {"outs": 3})),
    (r"병살|더블\s?플레이", lambda m: ("in_play", {"outs": 2})),
    # '2루타' '3루 타' '삼루타'처럼 숫자 표기·띄어쓰기가 달라도
    (r"안타|(?:[23]|이|삼)\s?루\s?타|홈런|실책", lambda m: ("in_play", {"outs": 0})),
    (r"(?:잡아냅니다|잡았습니다|처리합니다)?[,.\s]*아웃|잡아냅니다|잡았습니다|처리합니다|희생|야수\s?선택",
     lambda m: ("in_play", {"outs": 1})),
    # 투구 판정
    (r"파울|커트", lambda m: ("foul", {})),
    (r"헛\s?스윙|스트라이크", lambda m: ("strike", {})),
    (r"(?<![땅뜬])볼(?!넷|\s?카운트)|빠집니다|빠졌|골라", lambda m: ("ball", {})),
]
RULES = [(re.compile(p), f) for p, f in RULES]


def realign(text, words):
    """정렬 단위를 받아쓴 원문 위에 순서대로 맞춰서, 원문의 띄어쓰기·문장부호를 되살린 단어 목록을 만든다.
    각 조각 = (직전 단위 끝 ~ 이 단위 끝)까지의 원문, 시각은 이 단위의 시각.
    Qwen3-ForcedAligner는 '공격입니다'를 '공격' '입니다'로 쪼개므로 그냥 이어 붙이면 단어 경계가 틀어진다."""
    out, pos = [], 0
    for w in words:
        unit = w["text"].strip()
        i = text.find(unit, pos) if unit else -1
        if i < 0:  # 원문에서 못 찾은 단위는 건너뜀
            continue
        out.append({"text": text[pos:i + len(unit)], "start": w["start"], "end": w["end"]})
        pos = i + len(unit)
    return out


def words_of(cid):
    t = transcripts[cid]
    return realign(t["text"], t["words"]) if ASR == "qwen3" else t["words"]


def extract_actions(words):
    """단어 목록 → [(시각, 메서드, 인자, 걸린 글자)]. 시각은 걸린 글자가 끝나는 단어의 끝 시각"""
    text, word_end_at = "", []  # word_end_at[i] = 글자 i가 속한 단어의 끝 시각
    for w in words:
        text += w["text"]
        word_end_at += [w["end"]] * len(w["text"])

    used = [False] * len(text)
    actions = []
    for pattern, make in RULES:
        for m in pattern.finditer(text):
            if any(used[m.start():m.end()]):
                continue
            used[m.start():m.end()] = [True] * (m.end() - m.start())
            name, kwargs = make(m)
            actions.append((word_end_at[m.end() - 1], name, kwargs, m.group(0)))
    return sorted(actions, key=lambda a: a[0])


# 첫 클립: 복원된 단어 몇 개와 뽑힌 일 확인
cid = next(iter(transcripts))
print(words_of(cid)[:6])
for t, name, kwargs, hit in extract_actions(words_of(cid))[:15]:
    print(f"{t:6.2f}  {name:<10} {kwargs}  ← '{hit.strip()}'")

[{'text': '8회말', 'start': 0.56, 'end': 1.04}, {'text': ' 기아', 'start': 1.44, 'end': 1.76}, {'text': ' 공격', 'start': 1.76, 'end': 2.08}, {'text': '입니다', 'start': 2.08, 'end': 2.4}, {'text': '. 스코어', 'start': 2.72, 'end': 3.36}, {'text': ' 롯데', 'start': 3.36, 'end': 3.76}]
  8.40  set_outs   {'o': 0}  ← '노아웃'
  9.44  new_batter {}  ← '타석에'
 13.36  foul       {}  ← '파울'
 16.08  foul       {}  ← '파울'
 18.64  set_count  {'b': 0, 's': 2}  ← '노볼 투 스트라이크'
 21.20  ball       {}  ← '골라'
 24.40  set_count  {'b': 1, 's': 2}  ← '원볼 투 스트라이크'
 26.80  foul       {}  ← '파울'
 29.04  set_count  {'b': 1, 's': 2}  ← '원앤투'
 32.24  ball       {}  ← '볼'
 33.12  ball       {}  ← '골라'
 35.84  set_count  {'b': 3, 's': 2}  ← '풀카운트'
 40.96  in_play    {'outs': 1}  ← '잡았습니다'
 42.08  set_outs   {'o': 1}  ← '원아웃'
 43.04  new_batter {}  ← '타석에'


In [41]:
# 5) 예측 + 평가
pred = {}
for cid, ts in queries.items():
    actions = [(t, name, kwargs) for t, name, kwargs, _ in extract_actions(words_of(cid))]
    pred[cid] = answer_queries(actions, ts)

(OUT / "pred.json").write_text(json.dumps(pred, ensure_ascii=False), encoding="utf-8")
scores = evaluate(gt, pred, groups)
(OUT / "score.json").write_text(json.dumps({"cer": cer, **scores}, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"{ASR} / {DATA.name} / CER {cer:.2%}")
print(f"{'group':<12} {'n':>5} {'exact':>7} {'ball':>7} {'strike':>7} {'out':>7}")
for name, s in scores.items():
    print(f"{name:<12} {s['n']:>5} {s['exact']:>7.2%} {s['b']:>7.2%} {s['s']:>7.2%} {s['o']:>7.2%}")

qwen3 / dataset_pilot_v3 / CER 4.58%
group            n   exact    ball  strike     out
all            309  77.67%  89.64%  87.70%  88.03%
dev             59  77.97%  81.36%  96.61%  91.53%
dev/easy        31  96.77%  96.77%  96.77% 100.00%
dev/hard        28  57.14%  64.29%  96.43%  82.14%
test           149  81.21%  93.96%  89.26%  89.26%
test/easy       80  83.75% 100.00%  93.75%  83.75%
test/hard       69  78.26%  86.96%  84.06%  95.65%
train          101  72.28%  88.12%  80.20%  84.16%
train/easy      47  89.36%  97.87%  89.36% 100.00%
train/hard      54  57.41%  79.63%  72.22%  70.37%


In [42]:
# 6) 오답 분석: 틀린 질의 바로 앞에 무슨 말이 있었고 무엇을 뽑았는지
def show_errors(cid, limit=8):
    words = words_of(cid)
    actions = extract_actions(words)
    shown = 0
    for g, p in zip(gt[cid], pred[cid]):
        if (g["b"], g["s"], g["o"]) == (p["b"], p["s"], p["o"]):
            continue
        heard = "".join(w["text"] for w in words if g["t"] - 4 <= w["end"] <= g["t"])
        took = [f"{n}{kw or ''}" for t, n, kw, _ in actions if g["t"] - 4 <= t <= g["t"]]
        print(f"t={g['t']:6.2f} 정답 {g['b']}-{g['s']} {g['o']}아웃 / 예측 {p['b']}-{p['s']} {p['o']}아웃")
        print(f"         들은 말: {heard.strip()}")
        print(f"         뽑은 일: {took}")
        shown += 1
        if shown == limit:
            break


worst = min(queries, key=lambda c: evaluate({c: gt[c]}, pred)["all"]["exact"])
print("가장 많이 틀린 클립:", worst, groups[worst])
show_errors(worst)

가장 많이 틀린 클립: 20240430SKHH02024_07B ['train', 'train/hard']
t= 32.99 정답 1-1 0아웃 / 예측 1-2 0아웃
         들은 말: 하거든요. 직구 몸쪽 스트라이크
         뽑은 일: ['strike']
t= 39.39 정답 2-1 0아웃 / 예측 1-0 0아웃
         들은 말: 되죠. 낮게 빠졌고요. 카운트
         뽑은 일: ['ball']
t= 52.21 정답 1-1 0아웃 / 예측 1-0 0아웃
         들은 말: 볼입니다. 볼 카운트
         뽑은 일: ['ball']
t= 66.31 정답 1-2 1아웃 / 예측 2-0 3아웃
         들은 말: . 노볼 1스트라이크. 카운트 1-2. 예, 맞습니다. 자, 높은 볼이구요. 카운트 2-2. 타올. 2스트라이크 이후에는 아무래도 타자가 불리해지죠. 우익선상 빠집니다. 2루타. 투수 교체 마운드에 고효준. 원아웃 타석에는 노시환 낮게 빠집니다. 카운트 2볼. 선구안이 좋네요. 2-1입니다. 볼. 네, 그렇습니다. 1루로 걸어나갑니다. 그렇죠. 원아웃 타석에는 최은성 초구 헛스윙 나옵니다. 스트라이크 존을 넓게 쓰고 있어요. 자, 140킬로 직구 볼 헛스윙 볼 카운트 2-2. 중요한 순간입니다. 높게 빠졌고요. 선구안이 좋네요. 자, 방망이 헛돌면서 삼진. 투아웃 타석에는 안치용 루킹 스트라이크. 141킬로 직구 쳤습니다. 2루타. 투아웃 타석에는 황영무 초구 12
         뽑은 일: ["set_count{'b': 0, 's': 1}", "set_count{'b': 1, 's': 2}", "set_count{'b': 2, 's': 2}", "set_count{'b': 2, 's': 1}", "set_count{'b': 2, 's': 2}", "set_count{'b': 2, 's': 0}", "set_outs{'o': 1}", "set_outs{'o': 1}", "set_outs{'o': 2}", "